# Figure Reproduction: Distribution of Daily Maximum Temperatures Averaged Across CA ISO Region 
Reproduction of a figure for CEC<br>
Notebook creation date: Aug 1, 2023<br> <br>
Modification from original figure: 
- show daily max instead of daily min 
- show just one season (summer) or month (july) -- hottest season/month instead of entire year 
- extend time period out to 2045 

## Step 0: Import required libraries 
Here, we'll import the `climakitae` library, along with additional libraries used in the notebook.

In [ ]:
import climakitae as ck
import statsmodels.api as sm # Stats library for computing PDF 
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import calendar # Used for printing calendar names from integers (i.e 9 --> Sep) 

# Plotting setting for in-notebook display 
%config InlineBackend.figure_format = 'retina'
mpl.rcParams['figure.dpi'] = 100 # Sets figure size in the notebook

## Step 1: Read in daily maximum temperature 
We'll use the climakitae library to retrieve daily maximum WRF temperature from the cloud storage bucket.

First, we'll create an `Application` object, which can be used to select and retrieve data. 

In [ ]:
# Create the Application object 
app = ck.Application()

Next, we use the `Application` object to make our data settings.

In [ ]:
# Set app.selections 
app.selections.timescale = "daily" 
app.selections.resolution = "9 km" 
app.selections.scenario_historical = ["Historical Climate"] 
app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"]
app.selections.data_type = "Gridded" 
app.selections.downscaling_method = ["Dynamical"] 
app.selections.time_slice = (1980, 2100)
app.selections.area_subset = "CA Electric Balancing Authority Areas" 
app.selections.cached_area = "CALISO" 
app.selections.variable = "Maximum air temperature at 2m" 
app.selections.units = "degF"
app.selections.area_average = "Yes"

Next, we'll retrieve the data from the catalog. The retrieval function will return an xarray Dataset object. 

In [ ]:
# Retrieve data
tmax_daily= app.retrieve()
tmax_daily = tmax_daily.squeeze() # Remove singleton dimensions

## Step 2: Select the time period of interest for the plot 
Choose to a month or set of months to subset the data by. <br><br>
For example, you may chose to generate the PDF and plot for just September, the 9th month of the year, using code in the following format: <br>
`data.isel(time = data.time.dt.month.isin([9]))`
<br><br>Or, you may want to look a the summer months-- June, July, and August (months 6,7, & 8): <br> 
`data.isel(time = data.time.dt.month.isin([6,7,8]))`
<br><br>You could customize this subset of months however you want. For example, for California you may choose to extend the summer months through September, since September generally has high temperatures. 

In [ ]:
# Chose the months to subset by. Must be a list of integers, even if just looking at one month 
month_subset = [6,7,8] # June, July, August
# month_subset = [9] # September 

# Subset the data 
tmax_daily = tmax_daily.isel(time = tmax_daily.time.dt.month.isin(month_subset))

## Step 3: Select your simulation of interest 
First, we'll print out the simulation options. 

In [ ]:
print(tmax_daily.simulation.values) 

Next, chose **one** simulation from the list of options

In [ ]:
sim_name = "WRF_EC-Earth3-Veg_r1i1p1f1"
tmax_daily = tmax_daily.sel(simulation = sim_name)

## Step 4: Split the data into historical and future time periods 
Next, we'll split the data into a historical period and a future period, by subsetting the time dimension. 
<br> You can easily change the ranges for these periods by modifying the code below. 

In [ ]:
# Choose the time periods of interest 
future_period = ["1993","2007"]
historical_period = ["2008","2022"]

In [ ]:
# Split data into historical and future period 
tmax_historical = tmax_daily.sel(time=slice(historical_period[0], historical_period[1]))
tmax_future = tmax_daily.sel(time=slice(future_period[0], future_period[1]))

As a last step, we'll read the data into memory to make the PDF and plotting steps faster. 

In [ ]:
tmax_historical = app.load(tmax_historical)
tmax_future = app.load(tmax_future)

## Step 5: Compute the probability density function
First, we'll define a function that computes a probability density function. 

In [ ]:
def get_pdf(da): 
    """Compute probability density function 
    Applies statsmodels.api method to input DataArray 
    
    Args: 
        da (xr.DataArray): DataArray with only 1 dimension 
        
    Returns: 
        kde_da (xr.DataArray): PDF        
    """
    var_name = da.name # Use name of DataArray as dimension name 
    da = da.squeeze() # Squeeze out singleton dimensions 
    kde = sm.nonparametric.KDEUnivariate(da) # Compute PDF 
    kde.fit()
    kde_da = xr.DataArray(
        data = kde.density, 
        coords=[kde.support], 
        dims=[var_name], 
        name="Density"
    )
    return kde_da

Next, we'll apply that function to our data to generate the PDF.

In [ ]:
tmax_historical_pdf = get_pdf(tmax_historical) 
tmax_future_pdf = get_pdf(tmax_future) 

## Step 6: Create the plot 
Finally, we'll plot up the data using matplotlib.

In [ ]:
# Create figure and axis objects, and add PDF lines to it 
fig, axs = plt.subplots(1)
pl_future = tmax_future_pdf.plot(
    ax=axs, 
    label="({0})".format(", ".join(future_period)), 
    color="red"
)
pl_historical = tmax_historical_pdf.plot(
    ax=axs, 
    label="({0})".format(", ".join(historical_period)), 
    color="blue"
)

# Add plot decorators (title, legend) 
axs.legend(title="Period") 
plt.suptitle("Distribution of Daily Maximum Temperatures Averaged Across CA ISO Region")
plt.title("Time period: {0}".format(", ".join([calendar.month_abbr[mon_index] for mon_index in month_subset])))

# Display the figure in the notebook
plt.show()

Next, some code to export the figure:

In [ ]:
# Format the months used so that it can be used in the fig title 
months_list = [calendar.month_abbr[mon_index] for mon_index in month_subset]
figname = "cec_fig1_maxtemp_distribution_{0}.png".format("".join(months_list))

# Save the figure. Set a high dpi to improve figure quality
fig.savefig(figname, dpi=200)